In [1]:
#### 8 Feature ####

import pandas as pd
import numpy as np
from sklearn import datasets
from sklearn.feature_selection import mutual_info_classif

# ---------------------------------------------------------
# 1. Load and Prepare the Dataset
# ---------------------------------------------------------
print("Loading Breast Cancer dataset...")
cancer_dataset = datasets.load_breast_cancer()

# Create a DataFrame
df = pd.DataFrame(data=cancer_dataset.data, columns=cancer_dataset.feature_names)
df['TARGET'] = cancer_dataset.target

print(f"Dataset shape: {df.shape}")

# ---------------------------------------------------------
# 2. Separate Features and Target
# ---------------------------------------------------------
X = df.drop('TARGET', axis=1)
y = df['TARGET']

# ---------------------------------------------------------
# 3. Calculate Mutual Information (Information Gain)
# ---------------------------------------------------------
print("\nCalculating Mutual Information scores...")

# Calculate MI scores
# random_state ensures reproducibility, n_neighbors is used for the continuous variable estimation
mi_scores = mutual_info_classif(X, y, random_state=42, n_neighbors=3)

# Create a pandas Series for easy viewing, indexed by feature names, sorted highest to lowest
mi_result = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)

# ---------------------------------------------------------
# 4. Display the Results
# ---------------------------------------------------------
print("\n--- Mutual Information Scores (Ranked) ---")
print(mi_result)

# Optional: If you want to see just the names of the top 10 features
top_10_features = mi_result.head(8).index.tolist()
print("\n--- Top 8 Features based on MI ---")
for i, feature in enumerate(top_10_features):
    print(f"{i+1}. {feature}")

Loading Breast Cancer dataset...
Dataset shape: (569, 31)

Calculating Mutual Information scores...

--- Mutual Information Scores (Ranked) ---
worst perimeter            0.471842
worst area                 0.464313
worst radius               0.451230
mean concave points        0.438806
worst concave points       0.436255
mean perimeter             0.402361
mean concavity             0.375447
mean radius                0.362276
mean area                  0.360023
area error                 0.340759
worst concavity            0.315259
perimeter error            0.275614
radius error               0.249301
worst compactness          0.225211
mean compactness           0.213439
concave points error       0.125415
worst texture              0.120331
concavity error            0.117440
mean texture               0.096540
worst smoothness           0.095697
worst symmetry             0.095435
mean smoothness            0.079740
compactness error          0.073390
mean symmetry              0

In [2]:
features = df[[
    'worst perimeter', 
    'worst area', 
    'worst radius', 
    'mean concave points', 
    'worst concave points', 
    'mean perimeter', 
    'mean concavity', 
    'mean radius'
]]
label = df['TARGET']

In [3]:
#### Cross validation ####

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

# ---- Settings ----
n_splits = 5           # number of folds (change to 5, 10, etc.)
random_state = 42

# ---- Data: Assuming you have 'features' and 'label' defined ----
X = features                
y = label                   
# For demonstration, I will create some dummy data if you just copy-paste this.
# REMOVE the 3 lines below and use your actual X and y
# from sklearn.datasets import make_classification
# X, y = make_classification(n_samples=4024, n_features=15, random_state=42)

# ensure arrays
X_arr = X.values if isinstance(X, pd.DataFrame) else np.asarray(X)
y_arr = y.values if isinstance(y, (pd.Series, pd.DataFrame)) else np.asarray(y)

# ---- Define models (pipelines) ----
models = {
    'k-NN (k=99)': KNeighborsClassifier(n_neighbors=99),

    'DecisionTree (max_depth=3)': Pipeline([
        ('scaler', StandardScaler()),
        ('dt', DecisionTreeClassifier(max_depth=3, random_state=random_state))
    ]),

    'SVM (linear, C=1.0)': SVC(kernel='linear', C=1.0, probability=False)
}

# Define the cross-validation strategy
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

print("=== Cross-validated metrics (Per-Fold Details) ===")

# Iterate over each model
for name, model in models.items():
    print(f"\n=========================================")
    print(f"Evaluating Model: {name}")
    print(f"=========================================")
    
    # Lists to store metrics across all folds to calculate the mean/std later if needed
    train_accs, test_accs = [], []
    train_precs, test_precs = [], []
    train_recs, test_recs = [], []
    train_f1s, test_f1s = [], []
    
    # Iterate over each fold
    for fold_idx, (train_index, test_index) in enumerate(skf.split(X_arr, y_arr)):
        fold_num = fold_idx + 1
        
        # Split the data for this fold
        X_train, X_test = X_arr[train_index], X_arr[test_index]
        y_train, y_test = y_arr[train_index], y_arr[test_index]
        
        # Train the model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Calculate Accuracies
        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        train_prec = precision_score(y_train, y_train_pred, zero_division=0)
        test_prec = precision_score(y_test, y_test_pred, zero_division=0)
        
        train_rec = recall_score(y_train, y_train_pred, zero_division=0)
        test_rec = recall_score(y_test, y_test_pred, zero_division=0)
        
        train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
        test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
        
        # Append to lists
        
        # Calculate Confusion Matrices
        cm_train = confusion_matrix(y_train, y_train_pred)
        cm_test = confusion_matrix(y_test, y_test_pred)
        
        # Store accuracies for overall summary
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        train_precs.append(train_prec)
        test_precs.append(test_prec)
        train_recs.append(train_rec)
        test_recs.append(test_rec)
        train_f1s.append(train_f1)
        test_f1s.append(test_f1)
        
        # --- Print Results for this Fold ---
        print(f"\n--- Fold {fold_num} ---")
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Test Accuracy : {test_acc:.4f}")
        print(f"  Train Precision : {np.mean(train_precs):.4f} ± {np.std(train_precs):.4f}")
        print(f"  Test Precision  : {np.mean(test_precs):.4f} ± {np.std(test_precs):.4f}")
        print(f"  Train Recall    : {np.mean(train_recs):.4f} ± {np.std(train_recs):.4f}")
        print(f"  Test Recall     : {np.mean(test_recs):.4f} ± {np.std(test_recs):.4f}")
        print(f"  Train F1-Score  : {np.mean(train_f1s):.4f} ± {np.std(train_f1s):.4f}")
        print(f"  Test F1-Score   : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")
        print("  Train Confusion Matrix:")
        print(cm_train)
        print("  Test Confusion Matrix:")
        print(cm_test)
        
    # ---------------------------------------------------------
    # NEW: Expanded Overall Summary
    # ---------------------------------------------------------
    print(f"\n*** Overall Summary for {name} ***")
    
    # Helper to print mean ± std for each metric
    metrics_summary = [
        ("Accuracy ", train_accs, test_accs),
        ("Precision", train_precs, test_precs),
        ("Recall   ", train_recs, test_recs),
        ("F1-Score ", train_f1s, test_f1s)
    ]
    
    for metric_name, train_list, test_list in metrics_summary:
        print(f"  Mean Train {metric_name}: {np.mean(train_list):.4f} ± {np.std(train_list):.4f}")
        print(f"  Mean Test  {metric_name}: {np.mean(test_list):.4f} ± {np.std(test_list):.4f}")
        print("-" * 40)
    

=== Cross-validated metrics (Per-Fold Details) ===

Evaluating Model: k-NN (k=99)

--- Fold 1 ---
  Train Accuracy: 0.9143
  Test Accuracy : 0.9298
  Train Precision : 0.8946 ± 0.0000
  Test Precision  : 0.9200 ± 0.0000
  Train Recall    : 0.9790 ± 0.0000
  Test Recall     : 0.9718 ± 0.0000
  Train F1-Score  : 0.9349 ± 0.0000
  Test F1-Score   : 0.9452 ± 0.0000
  Train Confusion Matrix:
[[136  33]
 [  6 280]]
  Test Confusion Matrix:
[[37  6]
 [ 2 69]]

--- Fold 2 ---
  Train Accuracy: 0.9231
  Test Accuracy : 0.8772
  Train Precision : 0.8991 ± 0.0045
  Test Precision  : 0.8859 ± 0.0341
  Train Recall    : 0.9808 ± 0.0017
  Test Recall     : 0.9718 ± 0.0000
  Train F1-Score  : 0.9381 ± 0.0032
  Test F1-Score   : 0.9266 ± 0.0187
  Train Confusion Matrix:
[[139  30]
 [  5 281]]
  Test Confusion Matrix:
[[31 12]
 [ 2 69]]

--- Fold 3 ---
  Train Accuracy: 0.9121
  Test Accuracy : 0.8860
  Train Precision : 0.8973 ± 0.0044
  Test Precision  : 0.8730 ± 0.0333
  Train Recall    : 0.9790 ± 0

In [4]:
#### Cross validation ####

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

# ---- Settings ----
n_splits = 8           # number of folds (change to 5, 10, etc.)
random_state = 42

# ---- Data: Assuming you have 'features' and 'label' defined ----
X = features                
y = label                   
# For demonstration, I will create some dummy data if you just copy-paste this.
# REMOVE the 3 lines below and use your actual X and y
# from sklearn.datasets import make_classification
# X, y = make_classification(n_samples=4024, n_features=15, random_state=42)

# ensure arrays
X_arr = X.values if isinstance(X, pd.DataFrame) else np.asarray(X)
y_arr = y.values if isinstance(y, (pd.Series, pd.DataFrame)) else np.asarray(y)

# ---- Define models (pipelines) ----
models = {
    'k-NN (k=99)': KNeighborsClassifier(n_neighbors=99),

    'DecisionTree (max_depth=3)': Pipeline([
        ('scaler', StandardScaler()),
        ('dt', DecisionTreeClassifier(max_depth=3, random_state=random_state))
    ]),

    'SVM (linear, C=1.0)': SVC(kernel='linear', C=1.0, probability=False)
}

# Define the cross-validation strategy
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

print("=== Cross-validated metrics (Per-Fold Details) ===")

# Iterate over each model
for name, model in models.items():
    print(f"\n=========================================")
    print(f"Evaluating Model: {name}")
    print(f"=========================================")
    
    # Lists to store metrics across all folds to calculate the mean/std later if needed
    train_accs, test_accs = [], []
    train_precs, test_precs = [], []
    train_recs, test_recs = [], []
    train_f1s, test_f1s = [], []
    
    # Iterate over each fold
    for fold_idx, (train_index, test_index) in enumerate(skf.split(X_arr, y_arr)):
        fold_num = fold_idx + 1
        
        # Split the data for this fold
        X_train, X_test = X_arr[train_index], X_arr[test_index]
        y_train, y_test = y_arr[train_index], y_arr[test_index]
        
        # Train the model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Calculate Accuracies
        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        train_prec = precision_score(y_train, y_train_pred, zero_division=0)
        test_prec = precision_score(y_test, y_test_pred, zero_division=0)
        
        train_rec = recall_score(y_train, y_train_pred, zero_division=0)
        test_rec = recall_score(y_test, y_test_pred, zero_division=0)
        
        train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
        test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
        
        # Append to lists
        
        # Calculate Confusion Matrices
        cm_train = confusion_matrix(y_train, y_train_pred)
        cm_test = confusion_matrix(y_test, y_test_pred)
        
        # Store accuracies for overall summary
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        train_precs.append(train_prec)
        test_precs.append(test_prec)
        train_recs.append(train_rec)
        test_recs.append(test_rec)
        train_f1s.append(train_f1)
        test_f1s.append(test_f1)
        
        # --- Print Results for this Fold ---
        print(f"\n--- Fold {fold_num} ---")
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Test Accuracy : {test_acc:.4f}")
        print(f"  Train Precision : {np.mean(train_precs):.4f} ± {np.std(train_precs):.4f}")
        print(f"  Test Precision  : {np.mean(test_precs):.4f} ± {np.std(test_precs):.4f}")
        print(f"  Train Recall    : {np.mean(train_recs):.4f} ± {np.std(train_recs):.4f}")
        print(f"  Test Recall     : {np.mean(test_recs):.4f} ± {np.std(test_recs):.4f}")
        print(f"  Train F1-Score  : {np.mean(train_f1s):.4f} ± {np.std(train_f1s):.4f}")
        print(f"  Test F1-Score   : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")
        print("  Train Confusion Matrix:")
        print(cm_train)
        print("  Test Confusion Matrix:")
        print(cm_test)
        
    # ---------------------------------------------------------
    # NEW: Expanded Overall Summary
    # ---------------------------------------------------------
    print(f"\n*** Overall Summary for {name} ***")
    
    # Helper to print mean ± std for each metric
    metrics_summary = [
        ("Accuracy ", train_accs, test_accs),
        ("Precision", train_precs, test_precs),
        ("Recall   ", train_recs, test_recs),
        ("F1-Score ", train_f1s, test_f1s)
    ]
    
    for metric_name, train_list, test_list in metrics_summary:
        print(f"  Mean Train {metric_name}: {np.mean(train_list):.4f} ± {np.std(train_list):.4f}")
        print(f"  Mean Test  {metric_name}: {np.mean(test_list):.4f} ± {np.std(test_list):.4f}")
        print("-" * 40)
    

=== Cross-validated metrics (Per-Fold Details) ===

Evaluating Model: k-NN (k=99)

--- Fold 1 ---
  Train Accuracy: 0.9195
  Test Accuracy : 0.9167
  Train Precision : 0.9072 ± 0.0000
  Test Precision  : 0.9149 ± 0.0000
  Train Recall    : 0.9712 ± 0.0000
  Test Recall     : 0.9556 ± 0.0000
  Train F1-Score  : 0.9381 ± 0.0000
  Test F1-Score   : 0.9348 ± 0.0000
  Train Confusion Matrix:
[[154  31]
 [  9 303]]
  Test Confusion Matrix:
[[23  4]
 [ 2 43]]

--- Fold 2 ---
  Train Accuracy: 0.9116
  Test Accuracy : 0.9859
  Train Precision : 0.9020 ± 0.0052
  Test Precision  : 0.9463 ± 0.0314
  Train Recall    : 0.9712 ± 0.0000
  Test Recall     : 0.9778 ± 0.0222
  Train F1-Score  : 0.9353 ± 0.0028
  Test F1-Score   : 0.9618 ± 0.0270
  Train Confusion Matrix:
[[150  35]
 [  9 304]]
  Test Confusion Matrix:
[[26  1]
 [ 0 44]]

--- Fold 3 ---
  Train Accuracy: 0.9257
  Test Accuracy : 0.8310
  Train Precision : 0.9041 ± 0.0052
  Test Precision  : 0.9001 ± 0.0702
  Train Recall    : 0.9744 ± 0

In [5]:
#### Cross validation ####

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

# ---- Settings ----
n_splits = 10           # number of folds (change to 5, 10, etc.)
random_state = 42

# ---- Data: Assuming you have 'features' and 'label' defined ----
X = features                
y = label                   
# For demonstration, I will create some dummy data if you just copy-paste this.
# REMOVE the 3 lines below and use your actual X and y
# from sklearn.datasets import make_classification
# X, y = make_classification(n_samples=4024, n_features=15, random_state=42)

# ensure arrays
X_arr = X.values if isinstance(X, pd.DataFrame) else np.asarray(X)
y_arr = y.values if isinstance(y, (pd.Series, pd.DataFrame)) else np.asarray(y)

# ---- Define models (pipelines) ----
models = {
    'k-NN (k=99)': KNeighborsClassifier(n_neighbors=99),

    'DecisionTree (max_depth=3)': Pipeline([
        ('scaler', StandardScaler()),
        ('dt', DecisionTreeClassifier(max_depth=3, random_state=random_state))
    ]),

    'SVM (linear, C=1.0)': SVC(kernel='linear', C=1.0, probability=False)
}

# Define the cross-validation strategy
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

print("=== Cross-validated metrics (Per-Fold Details) ===")

# Iterate over each model
for name, model in models.items():
    print(f"\n=========================================")
    print(f"Evaluating Model: {name}")
    print(f"=========================================")
    
    # Lists to store metrics across all folds to calculate the mean/std later if needed
    train_accs, test_accs = [], []
    train_precs, test_precs = [], []
    train_recs, test_recs = [], []
    train_f1s, test_f1s = [], []
    
    # Iterate over each fold
    for fold_idx, (train_index, test_index) in enumerate(skf.split(X_arr, y_arr)):
        fold_num = fold_idx + 1
        
        # Split the data for this fold
        X_train, X_test = X_arr[train_index], X_arr[test_index]
        y_train, y_test = y_arr[train_index], y_arr[test_index]
        
        # Train the model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Calculate Accuracies
        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        train_prec = precision_score(y_train, y_train_pred, zero_division=0)
        test_prec = precision_score(y_test, y_test_pred, zero_division=0)
        
        train_rec = recall_score(y_train, y_train_pred, zero_division=0)
        test_rec = recall_score(y_test, y_test_pred, zero_division=0)
        
        train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
        test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
        
        # Append to lists
        
        # Calculate Confusion Matrices
        cm_train = confusion_matrix(y_train, y_train_pred)
        cm_test = confusion_matrix(y_test, y_test_pred)
        
        # Store accuracies for overall summary
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        train_precs.append(train_prec)
        test_precs.append(test_prec)
        train_recs.append(train_rec)
        test_recs.append(test_rec)
        train_f1s.append(train_f1)
        test_f1s.append(test_f1)
        
        # --- Print Results for this Fold ---
        print(f"\n--- Fold {fold_num} ---")
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Test Accuracy : {test_acc:.4f}")
        print(f"  Train Precision : {np.mean(train_precs):.4f} ± {np.std(train_precs):.4f}")
        print(f"  Test Precision  : {np.mean(test_precs):.4f} ± {np.std(test_precs):.4f}")
        print(f"  Train Recall    : {np.mean(train_recs):.4f} ± {np.std(train_recs):.4f}")
        print(f"  Test Recall     : {np.mean(test_recs):.4f} ± {np.std(test_recs):.4f}")
        print(f"  Train F1-Score  : {np.mean(train_f1s):.4f} ± {np.std(train_f1s):.4f}")
        print(f"  Test F1-Score   : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")
        print("  Train Confusion Matrix:")
        print(cm_train)
        print("  Test Confusion Matrix:")
        print(cm_test)
        
    # ---------------------------------------------------------
    # NEW: Expanded Overall Summary
    # ---------------------------------------------------------
    print(f"\n*** Overall Summary for {name} ***")
    
    # Helper to print mean ± std for each metric
    metrics_summary = [
        ("Accuracy ", train_accs, test_accs),
        ("Precision", train_precs, test_precs),
        ("Recall   ", train_recs, test_recs),
        ("F1-Score ", train_f1s, test_f1s)
    ]
    
    for metric_name, train_list, test_list in metrics_summary:
        print(f"  Mean Train {metric_name}: {np.mean(train_list):.4f} ± {np.std(train_list):.4f}")
        print(f"  Mean Test  {metric_name}: {np.mean(test_list):.4f} ± {np.std(test_list):.4f}")
        print("-" * 40)
    

=== Cross-validated metrics (Per-Fold Details) ===

Evaluating Model: k-NN (k=99)

--- Fold 1 ---
  Train Accuracy: 0.9199
  Test Accuracy : 0.9123
  Train Precision : 0.9049 ± 0.0000
  Test Precision  : 0.9167 ± 0.0000
  Train Recall    : 0.9752 ± 0.0000
  Test Recall     : 0.9429 ± 0.0000
  Train F1-Score  : 0.9387 ± 0.0000
  Test F1-Score   : 0.9296 ± 0.0000
  Train Confusion Matrix:
[[157  33]
 [  8 314]]
  Test Confusion Matrix:
[[19  3]
 [ 2 33]]

--- Fold 2 ---
  Train Accuracy: 0.9141
  Test Accuracy : 0.9649
  Train Precision : 0.9033 ± 0.0016
  Test Precision  : 0.9313 ± 0.0146
  Train Recall    : 0.9720 ± 0.0031
  Test Recall     : 0.9714 ± 0.0286
  Train F1-Score  : 0.9364 ± 0.0023
  Test F1-Score   : 0.9509 ± 0.0213
  Train Confusion Matrix:
[[156  34]
 [ 10 312]]
  Test Confusion Matrix:
[[20  2]
 [ 0 35]]

--- Fold 3 ---
  Train Accuracy: 0.9199
  Test Accuracy : 0.9123
  Train Precision : 0.9038 ± 0.0014
  Test Precision  : 0.9200 ± 0.0199
  Train Recall    : 0.9731 ± 0

In [6]:
#####  Optimal ######

import pandas as pd
import numpy as np
from sklearn import datasets
from sklearn.feature_selection import mutual_info_classif

# ---------------------------------------------------------
# 1. Load and Prepare the Dataset
# ---------------------------------------------------------
print("Loading Breast Cancer dataset...")
cancer_dataset = datasets.load_breast_cancer()

# Create a DataFrame
df = pd.DataFrame(data=cancer_dataset.data, columns=cancer_dataset.feature_names)
df['TARGET'] = cancer_dataset.target

print(f"Dataset shape: {df.shape}")

# ---------------------------------------------------------
# 2. Separate Features and Target
# ---------------------------------------------------------
X = df.drop('TARGET', axis=1)
y = df['TARGET']

# ---------------------------------------------------------
# 3. Calculate Mutual Information (Information Gain)
# ---------------------------------------------------------
print("\nCalculating Mutual Information scores...")

# Calculate MI scores
# random_state ensures reproducibility, n_neighbors is used for the continuous variable estimation
mi_scores = mutual_info_classif(X, y, random_state=42, n_neighbors=3)

# Create a pandas Series for easy viewing, indexed by feature names, sorted highest to lowest
mi_result = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)

# ---------------------------------------------------------
# 4. Display the Results
# ---------------------------------------------------------
print("\n--- Mutual Information Scores (Ranked) ---")
print(mi_result)

# Optional: If you want to see just the names of the top 10 features
top_10_features = mi_result.head(15).index.tolist()
print("\n--- Top 10 Features based on MI ---")
for i, feature in enumerate(top_10_features):
    print(f"{i+1}. {feature}")

Loading Breast Cancer dataset...
Dataset shape: (569, 31)

Calculating Mutual Information scores...

--- Mutual Information Scores (Ranked) ---
worst perimeter            0.471842
worst area                 0.464313
worst radius               0.451230
mean concave points        0.438806
worst concave points       0.436255
mean perimeter             0.402361
mean concavity             0.375447
mean radius                0.362276
mean area                  0.360023
area error                 0.340759
worst concavity            0.315259
perimeter error            0.275614
radius error               0.249301
worst compactness          0.225211
mean compactness           0.213439
concave points error       0.125415
worst texture              0.120331
concavity error            0.117440
mean texture               0.096540
worst smoothness           0.095697
worst symmetry             0.095435
mean smoothness            0.079740
compactness error          0.073390
mean symmetry              0

In [8]:
# Optimal selection: Top 15 features based on Mutual Information (Threshold > 0.20)
features = df[[
    'worst perimeter', 'worst area', 'worst radius', 'mean concave points',
    'worst concave points', 'mean perimeter', 'mean concavity', 'mean radius',
    'mean area', 'area error', 'worst concavity', 'perimeter error',
    'radius error', 'worst compactness', 'mean compactness'
]]
label = df['TARGET']

In [9]:
#### Cross validation ####

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

# ---- Settings ----
n_splits = 5           # number of folds (change to 5, 10, etc.)
random_state = 42

# ---- Data: Assuming you have 'features' and 'label' defined ----
X = features                
y = label                   
# For demonstration, I will create some dummy data if you just copy-paste this.
# REMOVE the 3 lines below and use your actual X and y
# from sklearn.datasets import make_classification
# X, y = make_classification(n_samples=4024, n_features=15, random_state=42)

# ensure arrays
X_arr = X.values if isinstance(X, pd.DataFrame) else np.asarray(X)
y_arr = y.values if isinstance(y, (pd.Series, pd.DataFrame)) else np.asarray(y)

# ---- Define models (pipelines) ----
models = {
    'k-NN (k=99)': KNeighborsClassifier(n_neighbors=99),

    'DecisionTree (max_depth=3)': Pipeline([
        ('scaler', StandardScaler()),
        ('dt', DecisionTreeClassifier(max_depth=3, random_state=random_state))
    ]),

    'SVM (linear, C=1.0)': SVC(kernel='linear', C=1.0, probability=False)
}

# Define the cross-validation strategy
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

print("=== Cross-validated metrics (Per-Fold Details) ===")

# Iterate over each model
for name, model in models.items():
    print(f"\n=========================================")
    print(f"Evaluating Model: {name}")
    print(f"=========================================")
    
    # Lists to store metrics across all folds to calculate the mean/std later if needed
    train_accs, test_accs = [], []
    train_precs, test_precs = [], []
    train_recs, test_recs = [], []
    train_f1s, test_f1s = [], []
    
    # Iterate over each fold
    for fold_idx, (train_index, test_index) in enumerate(skf.split(X_arr, y_arr)):
        fold_num = fold_idx + 1
        
        # Split the data for this fold
        X_train, X_test = X_arr[train_index], X_arr[test_index]
        y_train, y_test = y_arr[train_index], y_arr[test_index]
        
        # Train the model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Calculate Accuracies
        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        train_prec = precision_score(y_train, y_train_pred, zero_division=0)
        test_prec = precision_score(y_test, y_test_pred, zero_division=0)
        
        train_rec = recall_score(y_train, y_train_pred, zero_division=0)
        test_rec = recall_score(y_test, y_test_pred, zero_division=0)
        
        train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
        test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
        
        # Append to lists
        
        # Calculate Confusion Matrices
        cm_train = confusion_matrix(y_train, y_train_pred)
        cm_test = confusion_matrix(y_test, y_test_pred)
        
        # Store accuracies for overall summary
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        train_precs.append(train_prec)
        test_precs.append(test_prec)
        train_recs.append(train_rec)
        test_recs.append(test_rec)
        train_f1s.append(train_f1)
        test_f1s.append(test_f1)
        
        # --- Print Results for this Fold ---
        print(f"\n--- Fold {fold_num} ---")
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Test Accuracy : {test_acc:.4f}")
        print(f"  Train Precision : {np.mean(train_precs):.4f} ± {np.std(train_precs):.4f}")
        print(f"  Test Precision  : {np.mean(test_precs):.4f} ± {np.std(test_precs):.4f}")
        print(f"  Train Recall    : {np.mean(train_recs):.4f} ± {np.std(train_recs):.4f}")
        print(f"  Test Recall     : {np.mean(test_recs):.4f} ± {np.std(test_recs):.4f}")
        print(f"  Train F1-Score  : {np.mean(train_f1s):.4f} ± {np.std(train_f1s):.4f}")
        print(f"  Test F1-Score   : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")
        print("  Train Confusion Matrix:")
        print(cm_train)
        print("  Test Confusion Matrix:")
        print(cm_test)
        
    # ---------------------------------------------------------
    # NEW: Expanded Overall Summary
    # ---------------------------------------------------------
    print(f"\n*** Overall Summary for {name} ***")
    
    # Helper to print mean ± std for each metric
    metrics_summary = [
        ("Accuracy ", train_accs, test_accs),
        ("Precision", train_precs, test_precs),
        ("Recall   ", train_recs, test_recs),
        ("F1-Score ", train_f1s, test_f1s)
    ]
    
    for metric_name, train_list, test_list in metrics_summary:
        print(f"  Mean Train {metric_name}: {np.mean(train_list):.4f} ± {np.std(train_list):.4f}")
        print(f"  Mean Test  {metric_name}: {np.mean(test_list):.4f} ± {np.std(test_list):.4f}")
        print("-" * 40)
    

=== Cross-validated metrics (Per-Fold Details) ===

Evaluating Model: k-NN (k=99)

--- Fold 1 ---
  Train Accuracy: 0.9077
  Test Accuracy : 0.9211
  Train Precision : 0.8836 ± 0.0000
  Test Precision  : 0.9079 ± 0.0000
  Train Recall    : 0.9825 ± 0.0000
  Test Recall     : 0.9718 ± 0.0000
  Train F1-Score  : 0.9305 ± 0.0000
  Test F1-Score   : 0.9388 ± 0.0000
  Train Confusion Matrix:
[[132  37]
 [  5 281]]
  Test Confusion Matrix:
[[36  7]
 [ 2 69]]

--- Fold 2 ---
  Train Accuracy: 0.9099
  Test Accuracy : 0.8772
  Train Precision : 0.8838 ± 0.0002
  Test Precision  : 0.8756 ± 0.0323
  Train Recall    : 0.9843 ± 0.0017
  Test Recall     : 0.9789 ± 0.0070
  Train F1-Score  : 0.9313 ± 0.0009
  Test F1-Score   : 0.9239 ± 0.0148
  Train Confusion Matrix:
[[132  37]
 [  4 282]]
  Test Confusion Matrix:
[[30 13]
 [ 1 70]]

--- Fold 3 ---
  Train Accuracy: 0.9121
  Test Accuracy : 0.8860
  Train Precision : 0.8872 ± 0.0047
  Test Precision  : 0.8661 ± 0.0296
  Train Recall    : 0.9813 ± 0

In [10]:
#### Cross validation ####

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

# ---- Settings ----
n_splits = 8           # number of folds (change to 5, 10, etc.)
random_state = 42

# ---- Data: Assuming you have 'features' and 'label' defined ----
X = features                
y = label                   
# For demonstration, I will create some dummy data if you just copy-paste this.
# REMOVE the 3 lines below and use your actual X and y
# from sklearn.datasets import make_classification
# X, y = make_classification(n_samples=4024, n_features=15, random_state=42)

# ensure arrays
X_arr = X.values if isinstance(X, pd.DataFrame) else np.asarray(X)
y_arr = y.values if isinstance(y, (pd.Series, pd.DataFrame)) else np.asarray(y)

# ---- Define models (pipelines) ----
models = {
    'k-NN (k=99)': KNeighborsClassifier(n_neighbors=99),

    'DecisionTree (max_depth=3)': Pipeline([
        ('scaler', StandardScaler()),
        ('dt', DecisionTreeClassifier(max_depth=3, random_state=random_state))
    ]),

    'SVM (linear, C=1.0)': SVC(kernel='linear', C=1.0, probability=False)
}

# Define the cross-validation strategy
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

print("=== Cross-validated metrics (Per-Fold Details) ===")

# Iterate over each model
for name, model in models.items():
    print(f"\n=========================================")
    print(f"Evaluating Model: {name}")
    print(f"=========================================")
    
    # Lists to store metrics across all folds to calculate the mean/std later if needed
    train_accs, test_accs = [], []
    train_precs, test_precs = [], []
    train_recs, test_recs = [], []
    train_f1s, test_f1s = [], []
    
    # Iterate over each fold
    for fold_idx, (train_index, test_index) in enumerate(skf.split(X_arr, y_arr)):
        fold_num = fold_idx + 1
        
        # Split the data for this fold
        X_train, X_test = X_arr[train_index], X_arr[test_index]
        y_train, y_test = y_arr[train_index], y_arr[test_index]
        
        # Train the model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Calculate Accuracies
        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        train_prec = precision_score(y_train, y_train_pred, zero_division=0)
        test_prec = precision_score(y_test, y_test_pred, zero_division=0)
        
        train_rec = recall_score(y_train, y_train_pred, zero_division=0)
        test_rec = recall_score(y_test, y_test_pred, zero_division=0)
        
        train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
        test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
        
        # Append to lists
        
        # Calculate Confusion Matrices
        cm_train = confusion_matrix(y_train, y_train_pred)
        cm_test = confusion_matrix(y_test, y_test_pred)
        
        # Store accuracies for overall summary
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        train_precs.append(train_prec)
        test_precs.append(test_prec)
        train_recs.append(train_rec)
        test_recs.append(test_rec)
        train_f1s.append(train_f1)
        test_f1s.append(test_f1)
        
        # --- Print Results for this Fold ---
        print(f"\n--- Fold {fold_num} ---")
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Test Accuracy : {test_acc:.4f}")
        print(f"  Train Precision : {np.mean(train_precs):.4f} ± {np.std(train_precs):.4f}")
        print(f"  Test Precision  : {np.mean(test_precs):.4f} ± {np.std(test_precs):.4f}")
        print(f"  Train Recall    : {np.mean(train_recs):.4f} ± {np.std(train_recs):.4f}")
        print(f"  Test Recall     : {np.mean(test_recs):.4f} ± {np.std(test_recs):.4f}")
        print(f"  Train F1-Score  : {np.mean(train_f1s):.4f} ± {np.std(train_f1s):.4f}")
        print(f"  Test F1-Score   : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")
        print("  Train Confusion Matrix:")
        print(cm_train)
        print("  Test Confusion Matrix:")
        print(cm_test)
        
    # ---------------------------------------------------------
    # NEW: Expanded Overall Summary
    # ---------------------------------------------------------
    print(f"\n*** Overall Summary for {name} ***")
    
    # Helper to print mean ± std for each metric
    metrics_summary = [
        ("Accuracy ", train_accs, test_accs),
        ("Precision", train_precs, test_precs),
        ("Recall   ", train_recs, test_recs),
        ("F1-Score ", train_f1s, test_f1s)
    ]
    
    for metric_name, train_list, test_list in metrics_summary:
        print(f"  Mean Train {metric_name}: {np.mean(train_list):.4f} ± {np.std(train_list):.4f}")
        print(f"  Mean Test  {metric_name}: {np.mean(test_list):.4f} ± {np.std(test_list):.4f}")
        print("-" * 40)
    

=== Cross-validated metrics (Per-Fold Details) ===

Evaluating Model: k-NN (k=99)

--- Fold 1 ---
  Train Accuracy: 0.9095
  Test Accuracy : 0.9028
  Train Precision : 0.8915 ± 0.0000
  Test Precision  : 0.8958 ± 0.0000
  Train Recall    : 0.9744 ± 0.0000
  Test Recall     : 0.9556 ± 0.0000
  Train F1-Score  : 0.9311 ± 0.0000
  Test F1-Score   : 0.9247 ± 0.0000
  Train Confusion Matrix:
[[148  37]
 [  8 304]]
  Test Confusion Matrix:
[[22  5]
 [ 2 43]]

--- Fold 2 ---
  Train Accuracy: 0.8996
  Test Accuracy : 0.9859
  Train Precision : 0.8841 ± 0.0074
  Test Precision  : 0.9368 ± 0.0410
  Train Recall    : 0.9760 ± 0.0016
  Test Recall     : 0.9778 ± 0.0222
  Train F1-Score  : 0.9278 ± 0.0033
  Test F1-Score   : 0.9567 ± 0.0320
  Train Confusion Matrix:
[[142  43]
 [  7 306]]
  Test Confusion Matrix:
[[26  1]
 [ 0 44]]

--- Fold 3 ---
  Train Accuracy: 0.9237
  Test Accuracy : 0.8169
  Train Precision : 0.8905 ± 0.0108
  Test Precision  : 0.8887 ± 0.0758
  Train Recall    : 0.9787 ± 0

In [11]:
#### Cross validation ####

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)

# ---- Settings ----
n_splits = 10           # number of folds (change to 5, 10, etc.)
random_state = 42

# ---- Data: Assuming you have 'features' and 'label' defined ----
X = features                
y = label                   
# For demonstration, I will create some dummy data if you just copy-paste this.
# REMOVE the 3 lines below and use your actual X and y
# from sklearn.datasets import make_classification
# X, y = make_classification(n_samples=4024, n_features=15, random_state=42)

# ensure arrays
X_arr = X.values if isinstance(X, pd.DataFrame) else np.asarray(X)
y_arr = y.values if isinstance(y, (pd.Series, pd.DataFrame)) else np.asarray(y)

# ---- Define models (pipelines) ----
models = {
    'k-NN (k=99)': KNeighborsClassifier(n_neighbors=99),

    'DecisionTree (max_depth=3)': Pipeline([
        ('scaler', StandardScaler()),
        ('dt', DecisionTreeClassifier(max_depth=3, random_state=random_state))
    ]),

    'SVM (linear, C=1.0)': SVC(kernel='linear', C=1.0, probability=False)
}

# Define the cross-validation strategy
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

print("=== Cross-validated metrics (Per-Fold Details) ===")

# Iterate over each model
for name, model in models.items():
    print(f"\n=========================================")
    print(f"Evaluating Model: {name}")
    print(f"=========================================")
    
    # Lists to store metrics across all folds to calculate the mean/std later if needed
    train_accs, test_accs = [], []
    train_precs, test_precs = [], []
    train_recs, test_recs = [], []
    train_f1s, test_f1s = [], []
    
    # Iterate over each fold
    for fold_idx, (train_index, test_index) in enumerate(skf.split(X_arr, y_arr)):
        fold_num = fold_idx + 1
        
        # Split the data for this fold
        X_train, X_test = X_arr[train_index], X_arr[test_index]
        y_train, y_test = y_arr[train_index], y_arr[test_index]
        
        # Train the model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Calculate Accuracies
        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        train_prec = precision_score(y_train, y_train_pred, zero_division=0)
        test_prec = precision_score(y_test, y_test_pred, zero_division=0)
        
        train_rec = recall_score(y_train, y_train_pred, zero_division=0)
        test_rec = recall_score(y_test, y_test_pred, zero_division=0)
        
        train_f1 = f1_score(y_train, y_train_pred, zero_division=0)
        test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
        
        # Append to lists
        
        # Calculate Confusion Matrices
        cm_train = confusion_matrix(y_train, y_train_pred)
        cm_test = confusion_matrix(y_test, y_test_pred)
        
        # Store accuracies for overall summary
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        train_precs.append(train_prec)
        test_precs.append(test_prec)
        train_recs.append(train_rec)
        test_recs.append(test_rec)
        train_f1s.append(train_f1)
        test_f1s.append(test_f1)
        
        # --- Print Results for this Fold ---
        print(f"\n--- Fold {fold_num} ---")
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Test Accuracy : {test_acc:.4f}")
        print(f"  Train Precision : {np.mean(train_precs):.4f} ± {np.std(train_precs):.4f}")
        print(f"  Test Precision  : {np.mean(test_precs):.4f} ± {np.std(test_precs):.4f}")
        print(f"  Train Recall    : {np.mean(train_recs):.4f} ± {np.std(train_recs):.4f}")
        print(f"  Test Recall     : {np.mean(test_recs):.4f} ± {np.std(test_recs):.4f}")
        print(f"  Train F1-Score  : {np.mean(train_f1s):.4f} ± {np.std(train_f1s):.4f}")
        print(f"  Test F1-Score   : {np.mean(test_f1s):.4f} ± {np.std(test_f1s):.4f}")
        print("  Train Confusion Matrix:")
        print(cm_train)
        print("  Test Confusion Matrix:")
        print(cm_test)
        
    # ---------------------------------------------------------
    # NEW: Expanded Overall Summary
    # ---------------------------------------------------------
    print(f"\n*** Overall Summary for {name} ***")
    
    # Helper to print mean ± std for each metric
    metrics_summary = [
        ("Accuracy ", train_accs, test_accs),
        ("Precision", train_precs, test_precs),
        ("Recall   ", train_recs, test_recs),
        ("F1-Score ", train_f1s, test_f1s)
    ]
    
    for metric_name, train_list, test_list in metrics_summary:
        print(f"  Mean Train {metric_name}: {np.mean(train_list):.4f} ± {np.std(train_list):.4f}")
        print(f"  Mean Test  {metric_name}: {np.mean(test_list):.4f} ± {np.std(test_list):.4f}")
        print("-" * 40)
    

=== Cross-validated metrics (Per-Fold Details) ===

Evaluating Model: k-NN (k=99)

--- Fold 1 ---
  Train Accuracy: 0.9141
  Test Accuracy : 0.8947
  Train Precision : 0.8927 ± 0.0000
  Test Precision  : 0.8919 ± 0.0000
  Train Recall    : 0.9814 ± 0.0000
  Test Recall     : 0.9429 ± 0.0000
  Train F1-Score  : 0.9349 ± 0.0000
  Test F1-Score   : 0.9167 ± 0.0000
  Train Confusion Matrix:
[[152  38]
 [  6 316]]
  Test Confusion Matrix:
[[18  4]
 [ 2 33]]

--- Fold 2 ---
  Train Accuracy: 0.9062
  Test Accuracy : 0.9649
  Train Precision : 0.8887 ± 0.0039
  Test Precision  : 0.9189 ± 0.0270
  Train Recall    : 0.9798 ± 0.0016
  Test Recall     : 0.9714 ± 0.0286
  Train F1-Score  : 0.9321 ± 0.0029
  Test F1-Score   : 0.9444 ± 0.0278
  Train Confusion Matrix:
[[149  41]
 [  7 315]]
  Test Confusion Matrix:
[[20  2]
 [ 0 35]]

--- Fold 3 ---
  Train Accuracy: 0.9121
  Test Accuracy : 0.9123
  Train Precision : 0.8898 ± 0.0036
  Test Precision  : 0.9118 ± 0.0243
  Train Recall    : 0.9793 ± 0